In [ ]:
import os, re, sys, json, math, time, shutil, secrets, pathlib, platform, resource, threading, subprocess

PRESET = "stage1"

REPO_URL = "https://github.com/baobot26/VNU-dangkyhocphan-premium.git"
BASE_REF = "phase-1"
HEAD_REF = "main"
K6_VERSION = "0.54.0"

LEVELS = [100, 300, 1000, 3000, 10000]
REPS = 3
LOGIN_RATE = 12
LOGIN_WINDOW_S = None
SCRIPT = "full_scenario.js"
SECTION_COUNT = 36
WARMUP_VUS, WARMUP_S = 50, 20
LOGIN_VUS = 200
PROBE_VUS = None
BCRYPT_CHECK = False
VARIANTS = [("phase-1", BASE_REF, {}), ("head", HEAD_REF, {})]

A_ENV = {"SERVER_TOMCAT_MAX_CONNECTIONS": "20000", "SERVER_TOMCAT_ACCEPT_COUNT": "1000"}
B_ENV = {"ADMIT_LOGIN": "3", "ADMIT_WRITE": "24", "ADMIT_READ": "64"}
C_ENV = {"READ_CACHE_TTL_MS": "1000"}
KEPT_ENV = {}
B_PAIR = [("B-off", HEAD_REF, {}), ("B-on", HEAD_REF, B_ENV)]
PRESETS = {
    "stage1": dict(VARIANTS=[("phase-1", BASE_REF, {})], PROBE_VUS=max(LEVELS), BCRYPT_CHECK=True),
    "AA": dict(VARIANTS=[("phase-1", BASE_REF, {}), ("phase-1-b", BASE_REF, {})], LEVELS=[1000, 3000]),
    "A": dict(VARIANTS=[("phase-1", BASE_REF, {}), ("A", BASE_REF, A_ENV)], LEVELS=[max(LEVELS)]),
    "B-rush": dict(VARIANTS=B_PAIR, LEVELS=[3000, max(LEVELS)]),
    "B-login": dict(VARIANTS=B_PAIR, LEVELS=[500], SCRIPT="login.js"),
    "B-overlap": dict(VARIANTS=B_PAIR, LEVELS=[3000], LOGIN_WINDOW_S=30),
    "C": dict(VARIANTS=[("C-off", HEAD_REF, {}), ("C-on", HEAD_REF, C_ENV)], LEVELS=[1000, 3000]),
    "final": dict(VARIANTS=[("phase-1", BASE_REF, {}), ("head", HEAD_REF, KEPT_ENV)]),
}
globals().update(PRESETS.get(PRESET, {}))
STUDENTS = max(LEVELS + [PROBE_VUS or 0, 1000])

QC = {"p95_ms": 3000, "p99_ms": 5000, "err_pct": 1.0, "enroll201_p95_ms": 5000, "token_p95_ms": 2000}
K6_CPU_LIMIT, K6_RSS_LIMIT_MB = 150, 20000

WORK = pathlib.Path(os.environ.get("DKHP_WORK", "/tmp/dkhp"))
OUT_DIR = pathlib.Path(os.environ.get("DKHP_OUT", "/kaggle/working"))
SRC = WORK / "src"
RESULTS = WORK / "results"
CHARTS = RESULTS / "charts"
for d in (WORK / "jars", CHARTS, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

STUDENT_PASSWORD = "LoadTest2026"
OWNER_PASSWORD = secrets.token_hex(12)
API_DB_PASSWORD = secrets.token_hex(12)
JWT_SECRET = secrets.token_urlsafe(48)
BASE_URL = "http://127.0.0.1:8080/api/v1"

CPUS = len(os.sched_getaffinity(0))
NOFILE = resource.getrlimit(resource.RLIMIT_NOFILE)
resource.setrlimit(resource.RLIMIT_NOFILE, (NOFILE[1], NOFILE[1]))

def sh(cmd, check=True, env=None, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", text=True,
                       capture_output=True, env=env, cwd=cwd)
    if show:
        if r.stdout.strip(): print(r.stdout[-3000:])
        if r.stderr.strip(): print(r.stderr[-1500:])
    if check and r.returncode:
        raise RuntimeError(f"Lệnh lỗi ({r.returncode}): {cmd}\n{r.stderr[-1500:]}")
    return r

PG_ENV = {**os.environ, "PGHOST": "127.0.0.1", "PGPASSWORD": OWNER_PASSWORD}
T_SESSION = time.time()
print("Máy:", platform.platform(), "| CPU:", CPUS, "| nofile:", NOFILE)
print("PRESET:", PRESET, "| mức VU:", LEVELS, "| lần lặp:", REPS, "| kịch bản:", SCRIPT)
for name, ref, env in VARIANTS:
    print(f"  biến thể {name}: ref {ref}, env {env}")

In [ ]:
if not (SRC / ".git").exists():
    sh(f"git clone -q {REPO_URL} {SRC}")
sh("git fetch -q --tags origin '+refs/heads/*:refs/remotes/origin/*'", cwd=SRC)

def resolve(ref):
    for cand in (f"refs/tags/{ref}", f"refs/remotes/origin/{ref}", ref):
        r = sh(f"git rev-parse -q --verify '{cand}^{{commit}}'", cwd=SRC, check=False, show=False)
        if r.returncode == 0:
            return r.stdout.strip()
    raise RuntimeError(f"Không tìm thấy ref '{ref}' trên {REPO_URL}. Tag và nhánh phải được push trước khi đo.")

REFS = {ref: resolve(ref) for ref in {HEAD_REF, *(v[1] for v in VARIANTS)}}
sh(f"git checkout -q --detach {REFS[HEAD_REF]}", cwd=SRC)
sh(r"sed -i 's/\r$//' mvnw docker/initdb/*.sh; sed -i '1s/^\xef\xbb\xbf//' load/k6/accounts.json", cwd=SRC)
print("Kịch bản k6 và dữ liệu:", HEAD_REF, REFS[HEAD_REF][:7])
for name, ref, env in VARIANTS:
    print(f"Biến thể {name}: {ref} {REFS[ref][:7]}")

In [ ]:
sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "openjdk-17-jdk-headless postgresql postgresql-contrib unzip curl > /dev/null")
if not shutil.which("k6"):
    url = f"https://github.com/grafana/k6/releases/download/v{K6_VERSION}/k6-v{K6_VERSION}-linux-amd64.tar.gz"
    sh(f"curl -sL {url} | tar xz -C /tmp && mv /tmp/k6-v{K6_VERSION}-linux-amd64/k6 /usr/local/bin/k6")
sh("pip install -q psutil matplotlib pandas psycopg2-binary")
VERSIONS = {k: sh(c, show=False).stdout.strip() for k, c in
            {"java": "java -version 2>&1 | head -1", "k6": "k6 version", "postgres": "psql --version"}.items()}
print(VERSIONS)

In [ ]:
import psycopg2

sh("service postgresql start && sleep 3")
pathlib.Path("/tmp/role.sql").write_text(f"CREATE ROLE dkhp_app LOGIN SUPERUSER PASSWORD '{OWNER_PASSWORD}';\n")
os.chmod("/tmp/role.sql", 0o644)
sh("su postgres -c 'psql -f /tmp/role.sql' && su postgres -c 'createdb -O dkhp_app dkhp'")
INITDB = SRC / "docker" / "initdb"
for f in sorted(INITDB.glob("*.sql")):
    sh(f"psql -U dkhp_app -d dkhp -v ON_ERROR_STOP=1 -q -f {f.relative_to(SRC)}", env=PG_ENV, cwd=SRC)
role_env = {**PG_ENV, "POSTGRES_USER": "dkhp_app", "POSTGRES_DB": "dkhp",
            "DB_USER": "dkhp_api", "DB_PASSWORD": API_DB_PASSWORD}
for f in sorted(INITDB.glob("*.sh")):
    sh(f"sh {f.relative_to(SRC)}", env=role_env, cwd=SRC)
sh(f"psql -U dkhp_app -d dkhp -v ON_ERROR_STOP=1 -q -v pw='{STUDENT_PASSWORD}' "
   f"-v students={STUDENTS} -v period_status=OPEN -f docker/seed/seed_dev.sql", env=PG_ENV, cwd=SRC, show=False)

PG_DSN = dict(host="127.0.0.1", dbname="dkhp", user="dkhp_app", password=OWNER_PASSWORD)

def q(sql, args=None):
    conn = psycopg2.connect(**PG_DSN)
    conn.autocommit = True
    try:
        cur = conn.cursor()
        cur.execute(sql, args)
        return cur.fetchall() if cur.description else None
    finally:
        conn.close()

def reset_db():
    q("TRUNCATE enrollment, idempotency_record RESTART IDENTITY")
    q("UPDATE section SET enrolled_count = 0")
    q("VACUUM ANALYZE section, account")
    q("CHECKPOINT")

SEED = dict(zip(("students", "sections", "seats", "period_status"), q(
    "SELECT (SELECT count(*) FROM account WHERE role = 'STUDENT'), (SELECT count(*) FROM section), "
    "(SELECT sum(capacity) FROM section), (SELECT string_agg(status::text, ',') FROM registration_period)")[0]))
SEED = {k: int(v) if k != "period_status" else v for k, v in SEED.items()}
assert SEED["students"] >= STUDENTS, SEED
print(SEED)

In [ ]:
JARS = {}

def build_jar(ref):
    commit = REFS[ref]
    if commit not in JARS:
        tree = WORK / "build" / commit[:10]
        if not tree.exists():
            sh(f"git worktree add -q --detach {tree} {commit}", cwd=SRC)
        sh(r"sed -i 's/\r$//' mvnw && sh ./mvnw -B -ntp -q -DskipTests package", cwd=tree)
        jar = WORK / "jars" / f"{commit[:10]}.jar"
        shutil.copy(next((tree / "target").glob("course-registration-*.jar")), jar)
        JARS[commit] = jar
    return JARS[commit]

for name, ref, env in VARIANTS:
    jar = build_jar(ref)
    print(f"{name}: {ref} {REFS[ref][:7]} {jar.name} {jar.stat().st_size / 1e6:.1f} MB")

In [ ]:
import urllib.request, urllib.error

APP_ENV = {**os.environ,
    "DB_URL": "jdbc:postgresql://127.0.0.1:5432/dkhp", "DB_USER": "dkhp_api", "DB_PASSWORD": API_DB_PASSWORD,
    "JWT_SECRET": JWT_SECRET, "SERVER_ADDRESS": "127.0.0.1", "HEALTH_ALLOWED_IPS": "127.0.0.1,::1",
    "RATE_LIMIT_IP": "100000000", "RATE_LIMIT_ACCOUNT": "100000000", "RATE_LIMIT_LOGIN": "100000000"}
APP = {"proc": None, "log": None}

def api(method, path, body=None, token=None):
    req = urllib.request.Request(BASE_URL + path, method=method,
        data=json.dumps(body).encode() if body is not None else None,
        headers={"Content-Type": "application/json", **({"Authorization": "Bearer " + token} if token else {})})
    try:
        r = urllib.request.urlopen(req, timeout=10); return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()

def stop_app():
    proc = APP["proc"]
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill(); proc.wait()
    if APP["log"]:
        APP["log"].close()
    APP.update(proc=None, log=None)

def start_app(jar, env, log_path=None):
    stop_app()
    log_path = log_path or RESULTS / "app.log"
    APP["log"] = open(log_path, "w")
    APP["proc"] = subprocess.Popen(["java", "-Xmx2g", "-jar", str(jar)], env={**APP_ENV, **env},
                                   stdout=APP["log"], stderr=subprocess.STDOUT)
    for _ in range(90):
        if APP["proc"].poll() is not None:
            break
        try:
            if urllib.request.urlopen(BASE_URL + "/health", timeout=2).status == 200:
                status, body = api("POST", "/auth/login", {"code": "SV00001", "password": STUDENT_PASSWORD})
                assert status == 200, (status, body)
                return
        except (urllib.error.URLError, OSError):
            time.sleep(2)
    stop_app()
    raise RuntimeError(f"Backend không lên được, xem {log_path}")

start_app(build_jar(VARIANTS[0][1]), VARIANTS[0][2])
print("Backend của", VARIANTS[0][0], "sẵn sàng, pid", APP["proc"].pid)
stop_app()

In [ ]:
import numpy as np, pandas as pd, psutil

K6_DIR = SRC / "load" / "k6"
TREND = "avg,min,med,p(90),p(95),p(99),max,count"
STATUSES = [0, 201, 409, 422, 429, 500, 502, 503, 504]
REQ_LINE = re.compile(r"method=(\w+) status=(\d+) apiMs=(\d+) dbMs=(\d+)")
FAIL_LINE = re.compile(r"(databaseFailure|internalFailure)=(\w+)")
BLKS = "SELECT blks_hit, blks_read FROM pg_stat_database WHERE datname = 'dkhp'"
ACTIVITY = ("SELECT count(*) FILTER (WHERE datname = 'dkhp'), count(*) FILTER (WHERE datname = 'dkhp' AND state = 'active'), "
            "count(*) FILTER (WHERE wait_event_type = 'Lock') FROM pg_stat_activity WHERE pid <> pg_backend_pid()")
for d in ("k6", "resources", "applog", "invariants"):
    (RESULTS / d).mkdir(exist_ok=True)
ROWS = []

def window_s(vus):
    return LOGIN_WINDOW_S if LOGIN_WINDOW_S is not None else max(5, math.ceil(vus / LOGIN_RATE))

def script_text(script, seen):
    path = K6_DIR / script
    if script in seen or not path.exists():
        return ""
    seen.add(script)
    text = path.read_text()
    return text + "".join(script_text(dep, seen) for dep in re.findall(r"""from\s+['"]\./([\w.-]+\.js)['"]""", text))

def wrap(script):
    own = (K6_DIR / script).read_text()
    everything = script_text(script, set())
    th = {"http_reqs{phase:rush}": ["count>=0"], "http_req_duration{phase:rush}": ["max>=0"],
          "http_req_blocked{phase:rush}": ["max>=0"], "http_req_failed{phase:rush}": ["rate>=0"]}
    for method in ("GET", "POST", "DELETE"):
        th[f"http_req_duration{{phase:rush,method:{method}}}"] = ["max>=0"]
    for status in STATUSES:
        th[f"http_reqs{{status:{status}}}"] = ["count>=0"]
    if re.search(r"""['"]rejected['"]""", everything):
        for group in ("login", "write", "read"):
            th[f"rejected{{group:{group}}}"] = ["count>=0"]
    names = ["default"] + [n for n in ("setup", "teardown", "handleSummary")
                           if re.search(rf"export\s+(async\s+)?(function|const)\s+{n}\b", own)]
    (K6_DIR / f"_nb_{script}").write_text(
        f"import {{ options as own }} from './{script}';\n"
        f"export {{ {', '.join(names)} }} from './{script}';\n"
        f"const extra = {json.dumps(th)};\n"
        "export const options = Object.assign({}, own, { thresholds: Object.assign({}, extra, (own || {}).thresholds || {}) });\n")
    return f"_nb_{script}"

def k6_cmd(script, vus, window, extra=()):
    return ["k6", "run", "-q", "-e", f"BASE_URL={BASE_URL}", "-e", f"STUDENT_PASSWORD={STUDENT_PASSWORD}",
            "-e", f"VUS={vus}", "-e", f"LOGIN_WINDOW_S={window}", "-e", f"SECTION_COUNT={SECTION_COUNT}",
            *extra, f"load/k6/{script}"]

def warm_up(script):
    r = subprocess.run(k6_cmd(script, WARMUP_VUS, 3, ("--vus", str(WARMUP_VUS), "--duration", f"{WARMUP_S}s",
                                                      "--no-summary", "--no-thresholds")),
                       cwd=SRC, text=True, capture_output=True)
    if r.returncode:
        raise RuntimeError("Lượt làm nóng lỗi: " + (r.stdout + r.stderr)[-800:])

def cpu_s(procs):
    total = 0.0
    for p in procs:
        try:
            t = p.cpu_times(); total += t.user + t.system
        except psutil.Error:
            pass
    return total

def rss_mb(p):
    try:
        return p.memory_info().rss / 1e6
    except psutil.Error:
        return None

def sampler(stop, rows, k6_pid, t0):
    java, k6p = psutil.Process(APP["proc"].pid), psutil.Process(k6_pid)
    conn = psycopg2.connect(**PG_DSN); conn.autocommit = True; cur = conn.cursor()
    def snap():
        pgs = [p for p in psutil.process_iter(["name"]) if p.info["name"] == "postgres"]
        return time.time(), cpu_s([java]), cpu_s(pgs), cpu_s([k6p])
    psutil.cpu_percent(None); last = snap()
    while not stop.wait(1):
        now = snap(); dt = now[0] - last[0]
        cur.execute(ACTIVITY); conns, active, waiters = cur.fetchone()
        rows.append({"t": round(now[0] - t0, 1), "sys_cpu": psutil.cpu_percent(None),
                     "java_cpu": 100 * max(0, now[1] - last[1]) / dt, "pg_cpu": 100 * max(0, now[2] - last[2]) / dt,
                     "k6_cpu": 100 * max(0, now[3] - last[3]) / dt, "java_rss_mb": rss_mb(java), "k6_rss_mb": rss_mb(k6p),
                     "pg_conns": conns, "pg_active": active, "lock_waiters": waiters})
        last = now
    conn.close()

def run_k6(script, vus, window, run_id, extra=()):
    out = RESULTS / "k6" / f"{run_id}.json"
    rows, stop, t0 = [], threading.Event(), time.time()
    with open(RESULTS / "k6" / f"{run_id}.txt", "w") as log:
        proc = subprocess.Popen(k6_cmd(wrap(script), vus, window, ("--summary-trend-stats", TREND,
                                                                    "--summary-export", str(out), *extra)),
                                cwd=SRC, stdout=log, stderr=subprocess.STDOUT)
        th = threading.Thread(target=sampler, args=(stop, rows, proc.pid, t0), daemon=True); th.start()
        code = proc.wait()
    stop.set(); th.join()
    if not out.exists():
        raise RuntimeError(f"k6 không ghi kết quả (exit {code}), xem k6/{run_id}.txt")
    res = pd.DataFrame(rows, columns=["t", "sys_cpu", "java_cpu", "pg_cpu", "k6_cpu", "java_rss_mb", "k6_rss_mb",
                                      "pg_conns", "pg_active", "lock_waiters"])
    res.to_csv(RESULTS / "resources" / f"{run_id}.csv", index=False)
    return code, json.loads(out.read_text())["metrics"], res

def read_app_log(path, offset, keep):
    req, fails = {}, {}
    with open(path, errors="replace") as f, open(keep, "w") as out:
        f.seek(offset)
        for line in f:
            a = REQ_LINE.search(line)
            if a:
                r = req.setdefault((a[1], int(a[2])), [0, 0, 0])
                r[0] += 1; r[1] += int(a[3]); r[2] += int(a[4])
                continue
            out.write(line)
            b = FAIL_LINE.search(line)
            if b:
                fails[f"{b[1]}={b[2]}"] = fails.get(f"{b[1]}={b[2]}", 0) + 1
    return req, fails

def invariants_ok(run_id):
    r = sh("psql -U dkhp_app -d dkhp -f docker/verify/invariants.sql", env=PG_ENV, cwd=SRC, show=False, check=False)
    rows = [int(x) for x in re.findall(r"\((\d+) rows?\)", r.stdout)]
    ok = len(rows) == 7 and sum(rows) == 0
    if not ok:
        (RESULTS / "invariants" / f"{run_id}.txt").write_text(r.stdout + r.stderr)
    return ok

def mv(m, name, key):
    v = m.get(name, {}).get(key)
    return None if v is None else float(v)

def log_ms(req, pick, i):
    n = sum(v[0] for k, v in req.items() if pick(k))
    return sum(v[i] for k, v in req.items() if pick(k)) / n if n else None

def col(df, name, how):
    s = df[name].dropna()
    return float(getattr(s, how)()) if len(s) else None

def run_once(variant, vus, rep, tag="run", script=None, window=None, extra=()):
    name, ref, env = variant
    script = script or SCRIPT
    window = window_s(vus) if window is None else window
    run_id = f"{tag}-{name}-{vus}-r{rep}"
    full_log = RESULTS / "applog" / f"{run_id}.full.log"
    started = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
    start_app(build_jar(ref), env, full_log)
    try:
        reset_db(); warm_up(script); reset_db()
        offset = full_log.stat().st_size
        blk0 = q(BLKS)[0]
        code, m, res = run_k6(script, vus, window, run_id, extra)
        time.sleep(2)
        blk1 = q(BLKS)[0]
        inv = invariants_ok(run_id)
        split = {int(s): int(n) for s, n in q("SELECT response_status, count(*) FROM idempotency_record "
                                              "WHERE state = 'COMPLETED' GROUP BY 1")}
    finally:
        stop_app()
    req, fails = read_app_log(full_log, offset, RESULTS / "applog" / f"{run_id}.log")
    (RESULTS / "applog" / f"{run_id}.json").write_text(json.dumps(
        {f"{k[0]} {k[1]}": {"n": v[0], "apiMs": v[1], "dbMs": v[2]} for k, v in sorted(req.items())}, indent=1))
    full_log.unlink()

    total, rate = mv(m, "http_reqs", "count") or 0, mv(m, "http_reqs", "rate")
    if not total or not rate:
        raise RuntimeError(f"k6 không gửi được yêu cầu nào (exit {code}), xem k6/{run_id}.txt")
    span = total / rate
    rush = mv(m, "http_reqs{phase:rush}", "count") or 0
    sfx = "{phase:rush}" if rush else ""
    secs = max(span - window, 1) if rush else span
    n = rush or total
    d = "http_req_duration" + sfx
    busy = res[res.t >= window] if rush else res
    avg = mv(m, d, "avg")
    failed = mv(m, "http_req_failed" + sfx, "value")
    tokens = mv(m, "login_time_to_token_ms", "count")
    if tokens is None and not rush:
        tokens = total * (1 - (failed or 0)) if script == "login.js" else None
    e201 = mv(m, "enroll_time_to_201_ms", "count")
    e201 = split.get(201, 0) if e201 is None else e201
    done = sum(split.get(s, 0) for s in (201, 409, 422))
    rejected = mv(m, "rejected", "count")
    hit, read = blk1[0] - blk0[0], blk1[1] - blk0[1]
    k6_cpu, k6_rss = col(busy, "k6_cpu", "mean"), col(res, "k6_rss_mb", "max")
    enroll = lambda k: k[0] == "POST" and k[1] in (201, 409, 422)
    reads = lambda k: k[0] == "GET" and k[1] < 400
    return {
        "run_id": run_id, "tag": tag, "variant": name, "ref": ref, "commit": REFS[ref][:7], "env": json.dumps(env, sort_keys=True),
        "script": script, "vus": vus, "rep": rep, "window_s": window, "started_utc": started, "k6_exit": code,
        "scope": "rush" if rush else "all", "duration_s": round(span, 1), "measured_s": round(secs, 1),
        "requests": int(n), "rps": n / secs, "avg_ms": avg, "p50_ms": mv(m, d, "med"), "p95_ms": mv(m, d, "p(95)"),
        "p99_ms": mv(m, d, "p(99)"), "max_ms": mv(m, d, "max"), "in_flight": n / secs * avg / 1000 if avg is not None else None,
        "err_pct": None if failed is None else 100 * failed,
        "blocked_p95_ms": mv(m, "http_req_blocked", "p(95)"), "blocked_max_ms": mv(m, "http_req_blocked", "max"),
        "read_p95_ms": mv(m, "http_req_duration{phase:rush,method:GET}", "p(95)") if rush else None,
        "write_p95_ms": mv(m, "http_req_duration{phase:rush,method:POST}", "p(95)") if rush else None,
        "cancel_p95_ms": mv(m, "http_req_duration{phase:rush,method:DELETE}", "p(95)") if rush else None,
        "logins": tokens, "logins_per_s": None if tokens is None else tokens / (window if rush and window else span),
        "token_p50_ms": mv(m, "login_time_to_token_ms", "med"), "token_p95_ms": mv(m, "login_time_to_token_ms", "p(95)"),
        "enroll201_p50_ms": mv(m, "enroll_time_to_201_ms", "med"), "enroll201_p95_ms": mv(m, "enroll_time_to_201_ms", "p(95)"),
        "enroll_201": e201, "enroll_201_per_s": e201 / secs if done or e201 else None,
        "db_201": split.get(201, 0), "db_409": split.get(409, 0), "db_422": split.get(422, 0),
        "share_201_pct": 100 * split.get(201, 0) / done if done else None,
        "rejected": rejected, "rejected_pct": None if rejected is None else 100 * rejected / total,
        "rejected_login": mv(m, "rejected{group:login}", "count"), "rejected_write": mv(m, "rejected{group:write}", "count"),
        "rejected_read": mv(m, "rejected{group:read}", "count"),
        "replays": mv(m, "idempotent_replays", "count"), "replay_mismatch": mv(m, "replay_mismatch", "count"),
        "http_5xx": sum(mv(m, f"http_reqs{{status:{s}}}", "count") or 0 for s in (500, 502, 503, 504)),
        "no_response": mv(m, "http_reqs{status:0}", "count") or 0,
        "srv_5xx": sum(v[0] for k, v in req.items() if k[1] >= 500),
        "db_failures": sum(v for k, v in fails.items() if k.startswith("databaseFailure")),
        "internal_failures": sum(v for k, v in fails.items() if k.startswith("internalFailure")),
        "failures": json.dumps(fails, sort_keys=True),
        "sys_cpu": col(busy, "sys_cpu", "mean"), "java_cpu": col(busy, "java_cpu", "mean"), "pg_cpu": col(busy, "pg_cpu", "mean"),
        "k6_cpu": k6_cpu, "k6_rss_peak_mb": k6_rss, "java_rss_peak_mb": col(res, "java_rss_mb", "max"),
        "pg_conns_peak": col(res, "pg_conns", "max"), "pg_active_mean": col(busy, "pg_active", "mean"),
        "lock_wait_mean": col(busy, "lock_waiters", "mean"), "lock_wait_max": col(busy, "lock_waiters", "max"),
        "cache_hit_pct": 100 * hit / (hit + read) if hit + read else None,
        "enroll_api_ms": log_ms(req, enroll, 1), "enroll_db_ms": log_ms(req, enroll, 2),
        "read_api_ms": log_ms(req, reads, 1), "read_db_ms": log_ms(req, reads, 2),
        "login_api_ms": log_ms(req, lambda k: k == ("POST", 200), 1),
        "generator_limited": bool((k6_cpu or 0) > K6_CPU_LIMIT or (k6_rss or 0) > K6_RSS_LIMIT_MB),
        "invariants": inv, "error": None}

def record(variant, vus, rep, **kw):
    t0 = time.time()
    try:
        row = run_once(variant, vus, rep, **kw)
    except Exception as e:
        stop_app()
        row = {"run_id": f"{kw.get('tag', 'run')}-{variant[0]}-{vus}-r{rep}", "tag": kw.get("tag", "run"),
               "variant": variant[0], "vus": vus, "rep": rep, "error": repr(e)[:500]}
    row["wall_s"] = round(time.time() - t0)
    ROWS.append(row)
    pd.DataFrame(ROWS).to_csv(RESULTS / "runs.csv", index=False)
    return row

def num(v, fmt=",.0f"):
    return "-" if v is None or pd.isna(v) else format(v, fmt)

In [ ]:
LOGIN = {}
if LOGIN_RATE is None or BCRYPT_CHECK:
    probe = lambda vus, secs, tag: record(VARIANTS[0], vus, 0, tag=tag, script="login.js", window=0,
                                          extra=("--vus", str(vus), "--duration", f"{secs}s"))
    one, many = probe(1, 15, "login"), probe(LOGIN_VUS, 30, "login")
    assert not one["error"] and not many["error"], (one["error"], many["error"])
    LOGIN = {"hash_ms": one["p50_ms"], "logins_per_s": many["logins_per_s"],
             "theory_logins_per_s": CPUS * 1000 / one["p50_ms"], "login_vus": LOGIN_VUS,
             "p95_ms_at_login_vus": many["p95_ms"], "java_cpu_at_login_vus": many["java_cpu"]}
    print(f"Một lần đăng nhập ở 1 VU: {LOGIN['hash_ms']:.0f} ms (trung vị)")
    print(f"{LOGIN_VUS} VU: {LOGIN['logins_per_s']:.1f} đăng nhập/s, p95 {many['p95_ms']:.0f} ms, CPU Java {num(many['java_cpu'])} %")
    print(f"Lý thuyết {CPUS} nhân / thời gian băm: {LOGIN['theory_logins_per_s']:.1f} đăng nhập/s")
    if (many["java_cpu"] or 0) < 70 * CPUS:
        print(f"Java chưa dùng hết CPU ở {LOGIN_VUS} VU, nên số trên thấp hơn mức tối đa. Tăng LOGIN_VUS rồi đo lại.")
    if LOGIN_RATE is None:
        LOGIN_RATE = max(1, int(LOGIN["logins_per_s"] / 2))
    if BCRYPT_CHECK:
        original = q("SELECT password_hash FROM account WHERE code = 'SV00001'")[0][0]
        try:
            q("CREATE EXTENSION IF NOT EXISTS pgcrypto")
            q("UPDATE account SET password_hash = (SELECT crypt(%s, gen_salt('bf', 4))) WHERE role = 'STUDENT'", (STUDENT_PASSWORD,))
            fast = probe(LOGIN_VUS, 30, "login-cost4")
        finally:
            q("UPDATE account SET password_hash = %s WHERE role = 'STUDENT'", (original,))
            q("DROP EXTENSION IF EXISTS pgcrypto")
            q("VACUUM ANALYZE account")
        if not fast["error"]:
            LOGIN.update(cost4_logins_per_s=fast["logins_per_s"], cost4_p95_ms=fast["p95_ms"], cost4_java_cpu=fast["java_cpu"])
            print(f"Hash cost 4, {LOGIN_VUS} VU: {fast['logins_per_s']:.1f} đăng nhập/s, p95 {fast['p95_ms']:.0f} ms, CPU Java {num(fast['java_cpu'])} %")
LOGIN["login_rate"] = LOGIN_RATE
print("LOGIN_RATE =", LOGIN_RATE, "| cửa sổ đăng nhập theo mức:", {vus: window_s(vus) for vus in LEVELS})

In [ ]:
LIMITS = {k: sh(c, show=False, check=False).stdout.strip() for k, c in {
    "nofile_hard": "ulimit -Hn", "nofile_soft": "ulimit -n", "somaxconn": "sysctl -n net.core.somaxconn",
    "port_range": "sysctl -n net.ipv4.ip_local_port_range"}.items()}
LIMITS["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
print(LIMITS)
PROBE = None
if PROBE_VUS:
    PROBE = record(VARIANTS[0], PROBE_VUS, 0, tag="probe")
    if PROBE["error"]:
        print("Lượt thăm dò lỗi:", PROBE["error"])
    else:
        print(f"{PROBE_VUS} VU: k6 RSS đỉnh {num(PROBE['k6_rss_peak_mb'])} MB, k6 CPU {num(PROBE['k6_cpu'])} %, "
              f"Java CPU {num(PROBE['java_cpu'])} %, PostgreSQL CPU {num(PROBE['pg_cpu'])} %")
        print(f"req/s {PROBE['rps']:.1f}, p95 {num(PROBE['p95_ms'])} ms, lỗi {num(PROBE['err_pct'], '.2f')} %, "
              f"không có phản hồi {PROBE['no_response']:.0f}, blocked p95 {num(PROBE['blocked_p95_ms'])} ms")
        print("Máy phát tải là giới hạn: hạ mức cao nhất của LEVELS." if PROBE["generator_limited"]
              else "Máy phát tải chưa phải giới hạn ở mức này.")

In [ ]:
plan = [(rep, vus, v) for rep in range(1, REPS + 1) for vus in LEVELS for v in (VARIANTS if rep % 2 else VARIANTS[::-1])]
print(f"{len(plan)} lượt, ước tính {sum(window_s(vus) + 150 for _, vus, _ in plan) / 60:.0f} phút")
for i, (rep, vus, v) in enumerate(plan, 1):
    row = record(v, vus, rep)
    if row["error"]:
        print(f"[{i}/{len(plan)}] lần {rep} {vus:>5} VU {v[0]:<10} LỖI: {row['error'][:200]}")
        continue
    print(f"[{i}/{len(plan)}] lần {rep} {vus:>5} VU {v[0]:<10} p95={num(row['p95_ms']):>7} ms  {row['rps']:>7.1f} req/s  "
          f"enroll-201/s={num(row['enroll_201_per_s'], '.1f'):>6}  lỗi={num(row['err_pct'], '.2f')}%  "
          f"từ chối={num(row['rejected_pct'], '.1f')}%  bất biến={'PASS' if row['invariants'] else 'FAIL'}  ({row['wall_s']}s)")
all_runs = pd.DataFrame(ROWS)
runs = all_runs[(all_runs.tag == "run") & all_runs.error.isna()].copy()
assert len(runs), "Không có lượt đo nào thành công, xem runs.csv"

In [ ]:
SHOW = [("rps", "req/s"), ("p95_ms", "p95 (ms)"), ("p99_ms", "p99 (ms)"), ("err_pct", "Lỗi (%)"),
        ("blocked_p95_ms", "blocked p95 (ms)"), ("token_p95_ms", "Token p95 (ms)"),
        ("enroll201_p95_ms", "Tới 201 p95 (ms)"), ("enroll_201_per_s", "Enroll-201/s"),
        ("share_201_pct", "Tỷ lệ 201 (%)"), ("rejected_pct", "Bị từ chối (%)")]
NUMERIC = [c for c in runs.columns if pd.api.types.is_numeric_dtype(runs[c]) and c not in ("vus", "rep", "invariants", "generator_limited")]
TABLES = []

def spread(s, fmt=",.1f"):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return "-" if s.empty else f"{s.median():{fmt}} ({s.min():{fmt}}–{s.max():{fmt}})"

def md_table(title, df):
    lines = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    TABLES.append(f"### {title}\n\n" + "\n".join(lines) + "\n")

rows, capacity = [], {}
for (name, vus), g in runs.groupby(["variant", "vus"], sort=False):
    med = g.reindex(columns=list(QC)).apply(pd.to_numeric, errors="coerce").median()
    over = [k for k, lim in QC.items() if pd.notna(med[k]) and med[k] >= lim]
    if not g.invariants.all(): over.append("bất biến")
    if pd.to_numeric(g.replay_mismatch, errors="coerce").fillna(0).gt(0).any(): over.append("replay_mismatch")
    rows.append({"Biến thể": name, "VU": vus, "Lần": len(g), **{label: spread(g[k]) for k, label in SHOW},
                 "Bất biến": "PASS" if g.invariants.all() else "FAIL",
                 "Máy phát tải": "giới hạn" if g.generator_limited.any() else "ổn",
                 "Đạt D6": "ĐẠT" if not over else "KHÔNG ĐẠT (" + ", ".join(over) + ")"})
summary = pd.DataFrame(rows)
for name, g in summary.groupby("Biến thể", sort=False):
    ok = g.sort_values("VU")["Đạt D6"].eq("ĐẠT").cummin()
    capacity[name] = int(g.sort_values("VU").VU[ok].max()) if ok.any() else None
stats = runs.groupby(["variant", "vus"], sort=False)[NUMERIC].agg(["median", "min", "max"])
stats.columns = [f"{a}_{b}" for a, b in stats.columns]
stats.reset_index().to_csv(RESULTS / "summary.csv", index=False)
md_table("Tổng hợp: trung vị (nhỏ nhất – lớn nhất)", summary)
print("Mức chịu được theo biến thể:", capacity)
display(summary)

In [ ]:
METRICS = [("rps", "req/s", 1), ("enroll_201_per_s", "Enroll-201/s", 1), ("p95_ms", "p95 (ms)", -1), ("p99_ms", "p99 (ms)", -1),
           ("enroll201_p95_ms", "Tới 201 p95 (ms)", -1), ("token_p50_ms", "Token p50 (ms)", -1), ("token_p95_ms", "Token p95 (ms)", -1),
           ("logins_per_s", "Đăng nhập/s", 1), ("err_pct", "Lỗi (%)", -1), ("http_5xx", "Số 5xx", -1), ("no_response", "Không phản hồi", -1),
           ("blocked_p95_ms", "blocked p95 (ms)", -1), ("pg_cpu", "CPU PostgreSQL (%)", 0),
           ("share_201_pct", "Tỷ lệ 201 (%)", 0), ("rejected_pct", "Bị từ chối (%)", 0)]
base_name = VARIANTS[0][0]
rows = []
for vus in LEVELS:
    for name in [v[0] for v in VARIANTS[1:]]:
        for key, label, better in METRICS:
            pick = lambda variant: pd.to_numeric(runs[(runs.variant == variant) & (runs.vus == vus)][key], errors="coerce").dropna()
            b, v = pick(base_name), pick(name)
            if b.empty or v.empty:
                continue
            diff = v.median() - b.median()
            noise = max(b.max() - b.min(), v.max() - v.min())
            apart = v.min() > b.max() or v.max() < b.min()
            if min(len(b), len(v)) < 2: verdict = "thiếu lần lặp"
            elif better == 0: verdict = "tham khảo"
            elif abs(diff) <= noise or not apart: verdict = "trong sai lệch"
            else: verdict = "tốt hơn" if diff * better > 0 else "kém hơn"
            rows.append({"VU": vus, "Biến thể": name, "Chỉ số": label, f"Mốc ({base_name})": spread(b), "Biến thể đo được": spread(v),
                         "Thay đổi (%)": "-" if not b.median() else f"{100 * diff / b.median():+.1f}",
                         "Sai lệch": f"{noise:,.1f}", "Kết luận": verdict})
compare = pd.DataFrame(rows)
if len(compare):
    compare.to_csv(RESULTS / "compare.csv", index=False)
    md_table(f"So sánh với mốc {base_name}", compare)
    display(compare)
else:
    print("Chỉ có một biến thể, không có gì để so sánh.")

In [ ]:
done = all_runs[all_runs.error.isna()]
rows = [{"Loại lượt": r.tag, "Biến thể": r.variant, "VU": r.vus, "Lần": r.rep, "Dòng log": cls, "Số lần": cnt}
        for r in done.itertuples() for cls, cnt in json.loads(r.failures).items()]
five = (done.groupby(["tag", "variant", "vus"], sort=False)[["requests", "http_5xx", "no_response", "srv_5xx", "db_failures", "internal_failures"]]
        .sum().reset_index().rename(columns={"tag": "Loại lượt", "variant": "Biến thể", "vus": "VU", "requests": "Yêu cầu",
                                             "http_5xx": "5xx (k6)", "no_response": "Không phản hồi (k6)", "srv_5xx": "5xx (app.log)",
                                             "db_failures": "databaseFailure", "internal_failures": "internalFailure"}))
md_table("Lỗi 5xx theo biến thể và mức VU (cộng các lần lặp)", five)
display(five)
if rows:
    failures = pd.DataFrame(rows).groupby(["Loại lượt", "Biến thể", "VU", "Dòng log"], sort=False)["Số lần"].sum().reset_index()
    failures.to_csv(RESULTS / "failures.csv", index=False)
    md_table("Ngoại lệ trong app.log", failures)
    display(failures)
    f9 = failures["Dòng log"].eq("internalFailure=CannotCreateTransactionException").any()
    print("F9:", "có internalFailure=CannotCreateTransactionException, giả thuyết được xác nhận." if f9
          else "không thấy CannotCreateTransactionException; nguyên nhân 5xx là các lớp ở bảng trên.")
else:
    print("app.log không có dòng databaseFailure= hay internalFailure= nào.")

In [ ]:
COLS = {"rps": "req/s", "enroll_201_per_s": "Enroll-201/s", "in_flight": "in_flight", "p95_ms": "p95 (ms)",
        "java_cpu": "CPU Java (%)", "pg_cpu": "CPU PostgreSQL (%)", "k6_cpu": "CPU k6 (%)", "sys_cpu": "CPU máy (%)",
        "pg_active_mean": "Phiên DB chạy", "lock_wait_mean": "Chờ khóa TB", "lock_wait_max": "Chờ khóa đỉnh",
        "enroll_api_ms": "Enroll apiMs", "enroll_db_ms": "Enroll dbMs", "read_api_ms": "GET apiMs", "read_db_ms": "GET dbMs",
        "cache_hit_pct": "Cache hit (%)", "k6_rss_peak_mb": "k6 RSS đỉnh (MB)"}
base = runs[runs.variant == base_name]
bn = base.groupby("vus")[list(COLS)].agg(lambda s: pd.to_numeric(s, errors="coerce").median())
KNEE = None
if len(bn):
    bn.round(1).reset_index().to_csv(RESULTS / "bottleneck.csv", index=False)
    bottleneck = bn.round(1).rename(columns=COLS).reset_index().rename(columns={"vus": "VU"})
    md_table(f"Điểm nghẽn, biến thể {base_name} (trung vị)", bottleneck)
    display(bottleneck)
    rps = bn.rps.dropna()
    knee = int(next((a for a, b in zip(rps.index, rps.index[1:]) if rps[b] < 1.15 * rps[a]), rps.index.max()))
    top = bn.loc[bn.index.max()]
    KNEE = {"vus": knee, "rps": float(bn.rps[knee]), "in_flight": None if pd.isna(bn.in_flight[knee]) else float(bn.in_flight[knee])}
    print(f"Thông lượng ngừng tăng từ {knee} VU: {KNEE['rps']:.1f} req/s, khoảng {num(KNEE['in_flight'], '.1f')} yêu cầu đồng thời.")
    print(f"Ở {bn.index.max()} VU: Java {num(top.java_cpu)} %, PostgreSQL {num(top.pg_cpu)} %, k6 {num(top.k6_cpu)} % "
          f"trên {100 * CPUS} %; chờ khóa trung bình {num(top.lock_wait_mean, '.1f')}, đỉnh {num(top.lock_wait_max)}.")
    if pd.notna(top.enroll_api_ms) and top.enroll_api_ms:
        print(f"POST /enrollments: dbMs chiếm {100 * top.enroll_db_ms / top.enroll_api_ms:.0f} % của apiMs "
              f"({top.enroll_db_ms:.0f} / {top.enroll_api_ms:.0f} ms).")
else:
    print(f"Biến thể mốc {base_name} không có lượt đo nào thành công.")

In [ ]:
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
PALETTE = ["#4f6d9a", "#e9a23b", "#2a9d8f", "#d1495b", "#7b5ea7", "#6c757d"]
names = [v[0] for v in VARIANTS if v[0] in set(runs.variant)]
levels = sorted(runs.vus.unique())
xs = np.arange(len(levels))

def save(name):
    plt.tight_layout(); plt.savefig(CHARTS / name, dpi=130); plt.show(); plt.close()

def series(name, key):
    g = runs[runs.variant == name].assign(v=lambda d: pd.to_numeric(d[key], errors="coerce")).groupby("vus").v
    med, lo, hi = (g.agg(f).reindex(levels).to_numpy(float) for f in ("median", "min", "max"))
    return med, np.nan_to_num(np.vstack([med - lo, hi - med]))

def lines(ax, key, title, ylabel, limit=None, log=False):
    for i, name in enumerate(names):
        med, err = series(name, key)
        ax.errorbar(xs + (i - (len(names) - 1) / 2) * .06, med, yerr=err, marker="o", capsize=3, color=PALETTE[i % 6], label=name)
    if limit: ax.axhline(limit, color="#d1495b", ls=":", lw=1.2)
    if log: ax.set_yscale("log")
    ax.set_xticks(xs); ax.set_xticklabels(levels); ax.set_xlabel("VU"); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(fontsize=8)

def bars(ax, key, title, ylabel):
    w = .8 / len(names)
    for i, name in enumerate(names):
        med, err = series(name, key)
        ax.bar(xs + (i - (len(names) - 1) / 2) * w, np.nan_to_num(med), w, yerr=err, capsize=3, color=PALETTE[i % 6], label=name)
    ax.set_xticks(xs); ax.set_xticklabels(levels); ax.set_xlabel("VU"); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(fontsize=8)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
lines(axs[0], "p95_ms", "p95 giai đoạn đăng ký", "ms (thang log)", QC["p95_ms"], log=True)
lines(axs[1], "p99_ms", "p99 giai đoạn đăng ký", "ms (thang log)", QC["p99_ms"], log=True)
save("01-do-tre-theo-muc.png")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
bars(axs[0], "rps", "Thông lượng giai đoạn đăng ký", "req/s")
bars(axs[1], "enroll_201_per_s", "Đăng ký thành công mỗi giây", "enroll-201/s")
save("02-thong-luong.png")

fig, axs = plt.subplots(1, len(names), figsize=(6 * len(names), 4), squeeze=False)
for ax, name in zip(axs[0], names):
    bottom = np.zeros(len(levels))
    for j, (key, label) in enumerate([("java_cpu", "Java"), ("pg_cpu", "PostgreSQL"), ("k6_cpu", "k6")]):
        med = np.nan_to_num(series(name, key)[0])
        ax.bar(xs, med, .6, bottom=bottom, color=PALETTE[j], label=label); bottom += med
    ax.axhline(100 * CPUS, color="#d1495b", ls=":", lw=1.2)
    ax.set_xticks(xs); ax.set_xticklabels(levels); ax.set_xlabel("VU"); ax.set_ylabel("CPU % (100 = một nhân)")
    ax.set_title(f"CPU theo tiến trình: {name}"); ax.legend(fontsize=8)
save("03-cpu-theo-tien-trinh.png")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
lines(axs[0], "rejected_pct", "Tỷ lệ yêu cầu bị từ chối (429)", "%")
lines(axs[1], "err_pct", "Tỷ lệ lỗi giai đoạn đăng ký", "%", QC["err_pct"])
save("04-tu-choi-va-loi.png")

for name in names:
    first = runs[(runs.variant == name) & (runs.vus == levels[-1])].sort_values("rep").iloc[:1]
    if first.empty:
        continue
    run = first.iloc[0]
    res = pd.read_csv(RESULTS / "resources" / f"{run.run_id}.csv")
    if res.empty:
        continue
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
    for j, (key, label) in enumerate([("java_cpu", "Java"), ("pg_cpu", "PostgreSQL"), ("k6_cpu", "k6")]):
        axs[0].plot(res.t, res[key], color=PALETTE[j], label=label)
    axs[0].axvline(run.window_s, color="#6c757d", ls=":", lw=1.2)
    axs[0].set_xlabel("Giây"); axs[0].set_ylabel("CPU % (100 = một nhân)"); axs[0].legend(fontsize=8)
    axs[0].set_title(f"CPU theo thời gian: {run.run_id}")
    axs[1].plot(res.t, res.lock_waiters, color=PALETTE[3], label="Phiên chờ khóa")
    axs[1].plot(res.t, res.pg_active, color=PALETTE[0], label="Phiên đang chạy truy vấn")
    axs[1].axvline(run.window_s, color="#6c757d", ls=":", lw=1.2)
    axs[1].set_xlabel("Giây"); axs[1].set_ylabel("Số phiên"); axs[1].legend(fontsize=8); axs[1].set_title("PostgreSQL")
    save(f"05-tai-nguyen-{run.run_id}.png")
print(sorted(p.name for p in CHARTS.iterdir()))

In [ ]:
stop_app()
(RESULTS / "tables.md").write_text("\n".join(TABLES))
(RESULTS / "scripts").mkdir(exist_ok=True)
for f in K6_DIR.glob("*.js"):
    if not f.name.startswith("_nb_"):
        shutil.copy(f, RESULTS / "scripts" / f.name)
env_out = {
    "preset": PRESET, "repo": REPO_URL, "k6_ref": HEAD_REF, "refs": REFS,
    "variants": [{"name": n, "ref": r, "commit": REFS[r], "env": e} for n, r, e in VARIANTS],
    "levels": LEVELS, "reps": REPS, "script": SCRIPT, "students": STUDENTS, "seed": SEED,
    "login_rate": LOGIN_RATE, "login_window_s": {str(vus): window_s(vus) for vus in LEVELS},
    "warmup": {"vus": WARMUP_VUS, "seconds": WARMUP_S}, "qc": QC, "login": LOGIN, "limits": LIMITS,
    "probe": None if PROBE is None else {k: PROBE.get(k) for k in ("vus", "error", "k6_rss_peak_mb", "k6_cpu", "java_cpu", "pg_cpu", "rps",
                                                                 "p95_ms", "err_pct", "no_response", "blocked_p95_ms", "generator_limited")},
    "capacity": capacity, "knee": KNEE,
    "machine": {"os": platform.platform(), "cpu_count": CPUS, **VERSIONS},
    "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(T_SESSION)),
    "ended_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "session_s": round(time.time() - T_SESSION)}
(RESULTS / "env.json").write_text(json.dumps(env_out, indent=2, ensure_ascii=False, default=str))
zip_path = shutil.make_archive(str(OUT_DIR / "load_results"), "zip", RESULTS)
print("Đã đóng gói:", zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
print("Cả session:", round((time.time() - T_SESSION) / 60), "phút")